# JKO Steps for the Entropy Flow

This notebook generates `fig:gradflow-jko-entropy-steps`.  The JKO scheme
$$
    \rho^{k+1}\in\arg\min_\rho \frac{1}{2\tau}W_2^2(\rho^k,\rho)+\int \rho\log\rho
$$
is an implicit Euler scheme for the heat equation.  The second panel tracks inverse CDF values $Q_t(s)=F_t^{-1}(s)$ for fixed probability levels $s$.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks-figures":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks-figures"))

from figure_style import (
    BLUE,
    RED,
    VIOLET,
    ORANGE,
    GRAY,
    LIGHT_GRAY,
    DIRAC_MARKER_SIZE,
    box_axes,
    draw_point_clouds,
    draw_transport_segments,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()

NAME = "gradflow-jko-entropy-steps"
out = figure_dir(NAME)

The heat evolution is computed analytically for a two-bump Gaussian mixture.  Red-to-blue colors encode increasing JKO time.

In [2]:
x = np.linspace(-3.2, 3.2, 900)
weights = np.array([0.58, 0.42])
means = np.array([-1.12, 0.88])
base_var = np.array([0.055, 0.085])
times = np.array([0.0, 0.045, 0.11, 0.22, 0.42])

def gaussian_pdf(x, m, var):
    return np.exp(-0.5 * (x - m)**2 / var) / np.sqrt(2 * np.pi * var)

def heat_density(t):
    rho = sum(w * gaussian_pdf(x, m, v + 2 * t) for w, m, v in zip(weights, means, base_var))
    return rho / np.trapz(rho, x)

def cdf_from_density(rho):
    cdf = np.cumsum((rho[:-1] + rho[1:]) * 0.5 * np.diff(x))
    return np.r_[0, cdf] / cdf[-1]

densities = [heat_density(t) for t in times]
qs = np.linspace(0.08, 0.92, 15)
quantiles = []
for rho in densities:
    cdf = cdf_from_density(rho)
    quantiles.append(np.interp(qs, cdf, x))
quantiles = np.array(quantiles)

/var/folders/c3/8qf_y_jj6393y3l0dl0bb3k80000gp/T/ipykernel_94043/1045252550.py:12: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  return rho / np.trapz(rho, x)


In [3]:
fig, ax = plt.subplots(figsize=(3.10, 1.68))
for k, (t, rho) in enumerate(zip(times, densities)):
    color = interp_color(k / (len(times) - 1))
    ax.plot(x, rho, color=color, lw=1.15, alpha=0.95)
    ax.fill_between(x, 0, rho, color=color, alpha=0.052)
ax.set_xlim(-3.0, 3.0)
ax.set_ylim(0, max(densities[0]) * 1.10)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(labelsize=7, pad=1.5)
box_axes(ax)
save_pdf(fig, out / "density-steps.pdf", pad_inches=0.045)
plt.close(fig)

fig, ax = plt.subplots(figsize=(3.10, 1.62))
for j, s in enumerate(qs):
    shade = 0.35 + 0.45 * j / (len(qs) - 1)
    ax.plot(times, quantiles[:, j], color=VIOLET, lw=0.68, alpha=shade)
for k, t in enumerate(times):
    ax.scatter(np.full(len(qs), t), quantiles[k], s=DIRAC_MARKER_SIZE * 0.30, marker="o", color=interp_color(k/(len(times)-1)), edgecolor="none", linewidth=0, zorder=3)
ax.set_xlim(times[0] - 0.015, times[-1] + 0.015)
ax.set_ylim(-2.35, 2.35)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(labelsize=7, pad=1.5)
box_axes(ax)
save_pdf(fig, out / "quantile-steps.pdf", pad_inches=0.045)
plt.close(fig)